In [ ]:
# --- Standard Library ----------------------------------------------------------
from pathlib import Path
import sys

# --- Third-Party Libraries -----------------------------------------------------
from shapely.geometry import box
import pyrosm

# --- Local Application / Shared Code ------------------------------------------
%load_ext autoreload
%autoreload 2

# --- Custom Imports ------------------------------------------------------------
sys.path.append(str(Path().resolve().parent / 'share'))
import logging_utils as lu
import io_utils as iu
import folium_utils as fu
import plot_utils as pu
import dataframe_utils as du

In [ ]:
images_path = Path(r'C:\Users\joaqu\Dropbox\Apps\Overleaf\Analytics for Good\images') / 'Roads'
images_path.mkdir(parents=True, exist_ok=True)

In [ ]:
notebook_name = 'NL_roads.ipynb'

In [ ]:
# global definitions
_data_path = Path( '../shared_data/' )

In [ ]:
logger = lu.get_logger(notebook_name,_data_path / 'logs')

In [ ]:
# download if needed
country = 'Netherlands'
filepath = pyrosm.get_data(country, directory=_data_path)

In [ ]:
iu.get_remote_osm_pbf_timestamp(country)

In [ ]:
# Load the .pbf file
osm = pyrosm.OSM(filepath)

In [ ]:
osm_nodes, osm_edges = iu.load_or_acquire(
    _data_path,
    'nl_nodes_and_edges_all_modalities',
    osm.get_network,
    logger=logger,
    verbose_args=True,
    nodes=True, 
    network_type='all'
)

In [ ]:
optimized_nodes = du.optimize_dataframe(osm_nodes)
optimized_edges = du.optimize_dataframe(osm_edges)

In [ ]:
nodes, edges = iu.load_or_acquire(
    _data_path,
    'nl_optimized_nodes_and_edges_all_modalities',
    lambda: (optimized_nodes, optimized_edges),
    logger=logger,
    verbose_args=True
)

In [ ]:
du.assert_equal_ignoring_null_types(osm_edges, edges, check_dtype=False)

In [ ]:
du.assert_equal_ignoring_null_types(osm_nodes, nodes, check_dtype=False)

In [ ]:
du.plot_memory_usage_comparison(
    osm_nodes, 
    nodes, 
    per_column=True,
    file_name=images_path / 'nl_nodes_per_column_memory_usage.pdf'
)

In [ ]:
du.plot_memory_usage_comparison(
    osm_nodes, 
    nodes, 
    per_column=False,
    file_name=images_path / 'nl_nodes_total_memory_usage.pdf'
)

In [ ]:
du.plot_memory_usage_comparison(
    osm_edges, 
    edges, 
    per_column=True,
    file_name=images_path / 'nl_edges_per_column_memory_usage.pdf'
)

In [ ]:
du.plot_memory_usage_comparison(
    osm_edges, 
    edges, 
    per_column=False,
    file_name=images_path / 'nl_edges_total_memory_usage.pdf'
)

In [ ]:
edges.maxspeed.value_counts(dropna=False)

In [ ]:
m = edges[edges.maxspeed=='110'].explore(
    style_kwds={'weight': 6, 'color': 'purple'},
    tooltip='maxspeed',    
)
fu.folium_to_png(m, str(images_path / 'edges_maxspeed_110'))
m

In [ ]:
edges['highway'].value_counts(dropna=False)


In [ ]:
edges.osm_type.value_counts()

In [ ]:
print(nodes.crs)  # Check the CRS
print(edges.crs)  # Check the CRS

In [ ]:
import pyperclip
pyperclip.copy(f'{notebook_name} loaded with {nodes.shape[0]:,} nodes and {edges.shape[0]:,} edges.')
pyperclip.paste()

In [ ]:
edge_lengths = edges['length'].dropna().to_numpy()
pu.plot_fast_histogram(
    edge_lengths,
    title='edge lengths'
)
plt.savefig(images_path / 'nl_edges_length_histogram_all.pdf', bbox_inches='tight')
plt.show()

In [ ]:
import seaborn as sns
sns.violinplot(x=edge_lengths)
plt.savefig(images_path / 'nl_edges_length_violin.pdf', bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt 
sns.kdeplot(edge_lengths, bw_adjust=0.5)
plt.savefig(images_path / 'nl_edges_length_kde.pdf', bbox_inches='tight')
plt.xscale('log')
plt.savefig(images_path / 'nl_edges_length_kde_log.pdf', bbox_inches='tight')
plt.show()

In [ ]:
pyperclip.copy(edges['length'].describe().to_frame(name='length').T.to_latex(index=False, escape=True, float_format="%.1f"))


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter

# Select top edges by length
nlargest = 10_000
top = edges["length"].nlargest(nlargest)
rest = edges.loc[~edges.index.isin(top.index), "length"]

# Percentages
total = len(edges)
top_pct = 100 * len(top) / total
rest_pct = 100 * len(rest) / total

# Plot
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

rest.hist(bins=50, ax=ax[0])
ax[0].set_title(f"Remaining {len(rest):,} edges ({rest_pct:.1f}%)")
ax[0].set_xlabel("Length (m)")
ax[0].xaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))

top.hist(bins=50, ax=ax[1])
ax[1].set_title(f"Top {len(top):,} edges ({top_pct:.1f}%)")
ax[1].set_xlabel("Length (m)")
ax[1].xaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))

plt.tight_layout()
plt.savefig(images_path / 'nl_edges_length_histogram.pdf', bbox_inches='tight')
plt.show()


In [ ]:
m = edges.loc[[edges["length"].idxmax()]].explore(
    style_kwds={"color": "blue", "weight": 5},
    tooltip="length"
)
fu.folium_to_png(m, str(images_path / 'nl_edges_max_length'))
m

In [ ]:
import math
import mercantile
from shapely.geometry import box
from shapely.ops import unary_union

def tile_bounds_covering_bbox_autozoom(bounds, display_width=800, display_height=600):
    """
    Estimate the bounding box (xmin, ymin, xmax, ymax) of Web Mercator tiles
    that will cover the map after calling fit_bounds(bounds), assuming Leaflet
    auto-selects a zoom level to fit the map window.

    Args:
        bounds: (xmin, ymin, xmax, ymax) in WGS84 (EPSG:4326).
        display_width: Width of the map container in pixels.
        display_height: Height of the map container in pixels.

    Returns:
        Aligned bounding box (xmin, ymin, xmax, ymax) as a float tuple.
    """
    xmin, ymin, xmax, ymax = bounds

    # Estimate required zoom level to fit the bounding box inside a given display size
    # Based on Leaflet's default tile size = 256
    tile_size = 256
    world_bounds = 360  # total longitude span in degrees

    # Longitude span per pixel
    lon_span = xmax - xmin
    zoom_x = math.log2(world_bounds * display_width / (lon_span * tile_size))

    # Latitude span: we convert to Mercator meters
    def lat_to_merc(lat):
        lat = max(min(lat, 85.0511), -85.0511)
        return math.log(math.tan(math.radians(lat) / 2 + math.pi / 4))

    merc_min = lat_to_merc(ymin)
    merc_max = lat_to_merc(ymax)
    zoom_y = math.log2((2 * math.pi * 6378137) * display_height / ((merc_max - merc_min) * tile_size))

    zoom = int(min(zoom_x, zoom_y))

    # Get tile coverage at estimated zoom
    tiles = mercantile.tiles(xmin, ymin, xmax, ymax, zoom)
    tile_polys = [
        box(
            mercantile.bounds(t).west,
            mercantile.bounds(t).south,
            mercantile.bounds(t).east,
            mercantile.bounds(t).north
        )
        for t in tiles
    ]


    return unary_union(tile_polys).bounds


In [ ]:
lon = nodes.geometry.x.mean()
lat = nodes.geometry.y.mean()

In [ ]:
latDelta = 0.018/2
lonDelta = 0.029/2
first_bbox = (lon - lonDelta, lat - latDelta, lon + lonDelta, lat + latDelta)

In [ ]:
bbox = tile_bounds_covering_bbox_autozoom(first_bbox)
latDelta = 0.018
lonDelta = 0.029
bbox = (lon - lonDelta, lat - latDelta, lon + lonDelta, lat + latDelta)

In [ ]:
bboxPolygon = box(*bbox)

In [ ]:
# Keep only nodes inside bbox
nodesSubset = nodes[nodes.geometry.within(bboxPolygon)]

# Keep edges that intersect the bbox (even if one node is outside)
edgesSubset = edges[edges.geometry.intersects(bboxPolygon)]

In [ ]:
nodesSubset.shape, edgesSubset.shape

In [ ]:
import geopandas as gpd

pop_hdx = gpd.read_parquet(r"G:\My Drive\ABWBook\Population\HDX_NL.parquet")
pop_wp = gpd.read_parquet(r"G:\My Drive\ABWBook\Population\wp_nl.parquet")

In [ ]:
pop_hdx[pop_hdx.geometry.within(bboxPolygon)].Population.round(1).value_counts()

In [ ]:
subset_hdx = pop_hdx[pop_hdx.geometry.within(bboxPolygon)].copy()
subset_hdx["rounded_headcount"] = subset_hdx["Population"].round(1).astype("category")


In [ ]:
m = subset_hdx.explore(
    column="rounded_headcount",
    cmap="plasma",                         # or another color map
    marker_type="circle_marker",
    marker_kwds={"radius": 4, "fillOpacity": 0.7},
    tooltip="Population",
    name="Population categories",
    legend=True,
    categorical=True
)
#m = fu.adjust_bounds_for_points(m, subset[['latitude', 'longitude']])
m.fit_bounds(first_bbox)
fu.folium_to_png(m, str(images_path / 'nl_population_hdx_subset'))
m


In [ ]:
subset_wp = pop_wp[pop_wp.geometry.within(bboxPolygon)].copy()
subset_wp["rounded_headcount"] = subset_wp["Population"].round(-3).astype(int).astype("category")


In [ ]:
color_list = ["#e41a1c", "#377eb8", "#4daf4a"]  # red, blue, green

m = subset_wp.explore(
    column="rounded_headcount",
    cmap=color_list,                         # or another color map
    marker_type="circle_marker",
    marker_kwds={"radius": 5, "fillOpacity": 0.95},
    tooltip="Population",
    name="Population categories",
    legend=True,
    categorical=True
)
m.fit_bounds(first_bbox)
fu.folium_to_png(m, str(images_path / 'nl_population_wp_subset'))
m


In [ ]:
def explore_population_scaled_by_radius(
    gdf: gpd.GeoDataFrame,
    bbox_polygon,
    m: folium.Map = None,
    max_radius: float = 8.0,
    color: str = "red"
) -> folium.Map:
    """
    Display population points within a bounding polygon using marker radius
    scaled to the 'Population' column (not color-based).

    Args:
        gdf: GeoDataFrame with Point geometries and a 'Population' column.
        bbox_polygon: Shapely polygon to spatially filter.
        m: Optional folium.Map to overlay on.
        max_radius: Max circle marker radius in pixels.
        color: Marker stroke color.

    Returns:
        A folium.Map with population points styled by radius.
    """
    subset = gdf[gdf.geometry.within(bbox_polygon)].copy()
    if subset.empty:
        print("No population points within the bounding polygon.")
        return m or subset.explore()

    # Normalize radius
    pop_max = subset["Population"].max()
    subset["marker_radius"] = subset["Population"] / pop_max * max_radius

    # Build map manually using folium directly
    import folium
    from shapely.geometry import Point

    if m is None:
        # Center the map on the bbox centroid
        center = bbox_polygon.centroid
        m = folium.Map(location=[center.y, center.x], zoom_start=13)

    for _, row in subset.iterrows():
        if not isinstance(row.geometry, Point):
            continue  # skip invalid geometries

        lon, lat = row.geometry.x, row.geometry.y
        folium.CircleMarker(
            location=(lat, lon),
            radius=row.marker_radius,
            color=color,
            fill=True,
            fill_opacity=0.7,
            tooltip=f"Population: {row['Population']:,}",
        ).add_to(m)

    return m


In [ ]:
m = explore_population_scaled_by_radius(
    pop_hdx,
    bboxPolygon,
)
m.fit_bounds(first_bbox)
fu.folium_to_png(m, str(images_path / 'nl_population_hdx_radius_subset'))
m

In [ ]:
m = explore_population_scaled_by_radius(
    pop_wp,
    bboxPolygon,
)
m.fit_bounds(first_bbox)
fu.folium_to_png(m, str(images_path / 'nl_population_wp_radius_subset'))
m

In [ ]:
import geopandas as gpd
from shapely.geometry import box
from folium import LayerControl

# Create a GeoDataFrame from the bboxPolygon
bboxGDF = gpd.GeoDataFrame(geometry=[bboxPolygon], crs='EPSG:4326')

# First: edges in blue
m = edgesSubset.explore(
    color='blue',
    style_kwds={'weight': 5},  # This sets the linewidth
    name='Edges',
    tooltip=False
)

# Second: nodes in red
nodesSubset.explore(
    m=m,
    color='red',
    marker_kwds={'radius': 2},
    name='Nodes'
)

# Third: bounding box in black (transparent fill)
bboxGDF.explore(
    m=m,
    name='Bounding Box',
    color='black',
    fill=False,
    style_kwds={'weight': 8}  # This sets the linewidth
)

# Optional: add layer control
LayerControl(collapsed=False).add_to(m)
m.fit_bounds(first_bbox)
fu.folium_to_png(m, str( images_path / 'detail_center_box' ) )
m


In [ ]:
# Keep only rows where 'highway' is not null
edgesVisible = edgesSubset[edgesSubset['highway'].notna()].copy()

# Ensure 'highway' is a category *only with the present values*
edgesVisible['highway'] = edgesVisible['highway'].astype('category')
edgesVisible['highway'] = edgesVisible['highway'].cat.remove_unused_categories()

m = edgesVisible.explore(
    column='highway',
    categorical=True,
    legend=True,
    cmap='tab20',
    style_kwds={'weight': 5},
    name='Edges by highway',
    location=(52.16952247319173, 5.45816115915332),
    zoom_start=18,
    tiles='CartoDB positron'
)
nodesSubset.explore(
    m=m,
    color='red',
    marker_kwds={'radius': 2},
    name='Nodes'
)

fu.folium_to_png(m, str( images_path / 'detail_roundabout' ) )
m


In [ ]:
m = edgesVisible.explore(
    column='highway',
    categorical=True,
    legend=True,
    cmap='tab20',
    style_kwds={'weight': 5},
    name='Edges by highway',
    location=(52.165, 5.45816115915332),
    zoom_start=16.5,
    tiles='CartoDB positron'
)
nodesSubset.explore(
    m=m,
    color='red',
    marker_kwds={'radius': 2},
    name='Nodes'
)
m = subset_hdx.explore(
    m=m,
    column="rounded_headcount",
    cmap="Set1",                         # or another color map
    marker_type="circle_marker",
    marker_kwds={"radius": 5, "fillOpacity": 0.9},
    tooltip="Population",
    name="Population categories",
    legend=True,
    categorical=True
)
m

In [ ]:
from scipy.spatial import cKDTree
import numpy as np
import geopandas as gpd

def assign_nearest_nodes(
    pop_gdf: gpd.GeoDataFrame,
    nodes_gdf: gpd.GeoDataFrame,
    geometry_column: str = 'geometry',
    node_id_column: str = 'id',
    prefix: str = 'nearest_node'
) -> gpd.GeoDataFrame:
    """
    Assigns each point in `pop_gdf` the nearest node from `nodes_gdf`.

    Adds:
        - '{prefix}_id': ID of nearest node
        - '{prefix}_geom': Geometry of nearest node (Point)
        - '{prefix}_distance': Distance in CRS units (e.g., meters)

    Assumes both GeoDataFrames are in the same projected CRS.
    """
    if pop_gdf.crs != nodes_gdf.crs:
        raise ValueError('CRS mismatch between population and nodes')
    if not pop_gdf.crs.is_projected:
        raise ValueError('CRS must be projected (not geographic)')

    # Extract node coordinates and build KDTree
    node_coords = np.column_stack((nodes_gdf.geometry.x, nodes_gdf.geometry.y))
    tree = cKDTree(node_coords)

    # Query nearest node for each population point
    pop_coords = np.column_stack((pop_gdf[geometry_column].x, pop_gdf[geometry_column].y))
    distances, indices = tree.query(pop_coords, k=1)

    # Assign results
    nearest_ids = nodes_gdf.iloc[indices][node_id_column].values
    nearest_geoms = nodes_gdf.iloc[indices].geometry.values

    result = pop_gdf.copy()
    result[f'{prefix}_id'] = nearest_ids
    result[f'{prefix}_geom'] = nearest_geoms
    result[f'{prefix}_distance'] = distances

    return result


In [ ]:
subset_hdx = pop_hdx[pop_hdx.geometry.within(bboxPolygon)].copy()
subset_hdx["rounded_headcount"] = subset_hdx["Population"].round(1).astype("category")

nodesSubset = nodes[nodes.geometry.within(bboxPolygon)]

target_crs = 'EPSG:28992'
pop_hdx_proj = subset_hdx.to_crs(target_crs)
nodesSubset_proj = nodesSubset.to_crs(target_crs)

assigned = assign_nearest_nodes(
    pop_gdf=pop_hdx_proj,
    nodes_gdf=nodesSubset_proj,
    prefix='nearest_node_hdx'
)

In [ ]:
from folium import FeatureGroup, Marker
from folium.plugins import BeautifyIcon
from shapely.geometry import LineString, Point
from shapely.ops import transform
import pyproj

# Define transformer from EPSG:28992 → EPSG:4326
to_wgs84 = pyproj.Transformer.from_crs('EPSG:28992', 'EPSG:4326', always_xy=True).transform

# 1. Create map with base layers
m = edgesVisible.explore(
    tiles='CartoDB positron',
    column='highway',
    categorical=True,
    legend=True,
    cmap='tab20',
    style_kwds={'weight': 5},
    name='Edges by highway',
    location=(52.16952247319173, 5.45816115915332),
    zoom_start=18,
)

# Patch basemap layer name
for child in m._children.values():
    if hasattr(child, 'tiles') and 'cartocdn.com/light_all' in child.tiles:
        child.layer_name = 'Snapped population and nodes'

nodesSubset.explore(
    m=m,
    color='red',
    marker_kwds={'radius': 3},
    name='Nodes'
)

# 2. Add FeatureGroups for population and snap lines
pop_layer = FeatureGroup(name='Population markers')
snap_layer = FeatureGroup(name='Snap lines')

for _, row in assigned.iterrows():
    pop_geom = transform(to_wgs84, row['geometry'])
    node_geom = transform(to_wgs84, row['nearest_node_hdx_geom'])

    if isinstance(pop_geom, Point) and isinstance(node_geom, Point):
        line = LineString([pop_geom, node_geom])
        folium.PolyLine(
            locations=[(lat, lon) for lon, lat in line.coords],
            color='black',
            weight=2,
            dash_array='5,5',
            tooltip=f"Snap: {row['nearest_node_hdx_distance']:.1f} m"
        ).add_to(snap_layer)

        folium.Marker(
            location=(pop_geom.y, pop_geom.x),
            tooltip=f"Pop: {int(row.Population):,}",
            icon=BeautifyIcon(
                number=int(row.Population),
                icon_shape='circle',
                background_color='darkred',
                border_color='black',
                text_color='white',
                inner_icon_style='font-size:10px;',
                icon='user'
            )
        ).add_to(pop_layer)

# 3. Add both groups to the map
snap_layer.add_to(m)
pop_layer.add_to(m)

# 4. Enable full layer control
folium.LayerControl(collapsed=False).add_to(m)

fu.folium_to_png(m, str( images_path / 'detail_roundabout_hdx_pop_snap' ) )
m

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

# Step 1: Group by snapped node ID
grouped = (
    assigned.groupby('nearest_node_hdx_id')
    .agg({
        'Population': 'sum',
        'geometry': lambda geoms: Point(
            sum(p.x for p in geoms) / len(geoms),
            sum(p.y for p in geoms) / len(geoms)
        )
    })
    .reset_index()
)

# Step 2: Create new GeoDataFrame
agg_pop = gpd.GeoDataFrame(grouped, geometry='geometry', crs=assigned.crs)

# Step 3: Re-snap to nodes
resnapped = assign_nearest_nodes(
    pop_gdf=agg_pop,
    nodes_gdf=nodesSubset_proj,
    prefix='nearest_node_agg'
)


In [ ]:
from folium import FeatureGroup, Marker
from folium.plugins import BeautifyIcon
from shapely.geometry import LineString, Point
from shapely.ops import transform
import pyproj

# Step 1: Define transformer to EPSG:4326
to_wgs84 = pyproj.Transformer.from_crs(resnapped.crs, 'EPSG:4326', always_xy=True).transform

# 1. Create map with base layers
m = edgesVisible.explore(
    tiles='CartoDB positron',
    column='highway',
    categorical=True,
    legend=True,
    cmap='tab20',
    style_kwds={'weight': 5},
    name='Edges by highway',
    location=(52.16952247319173, 5.45816115915332),
    zoom_start=18,
)

# Patch basemap layer name
for child in m._children.values():
    if hasattr(child, 'tiles') and 'cartocdn.com/light_all' in child.tiles:
        child.layer_name = 'Snapped population and nodes'

nodesSubset.explore(
    m=m,
    color='red',
    marker_kwds={'radius': 3},
    name='Nodes'
)

# 2. Add FeatureGroups for population and snap lines
pop_layer = FeatureGroup(name='Population markers')
snap_layer = FeatureGroup(name='Snap lines')

for _, row in assigned.iterrows():
    pop_geom = transform(to_wgs84, row['geometry'])
    node_geom = transform(to_wgs84, row['nearest_node_hdx_geom'])

    if isinstance(pop_geom, Point) and isinstance(node_geom, Point):
        line = LineString([pop_geom, node_geom])
        folium.PolyLine(
            locations=[(lat, lon) for lon, lat in line.coords],
            color='black',
            weight=2,
            dash_array='5,5',
            tooltip=f"Snap: {row['nearest_node_hdx_distance']:.1f} m"
        ).add_to(snap_layer)

        folium.Marker(
            location=(pop_geom.y, pop_geom.x),
            tooltip=f"Pop: {int(row.Population):,}",
            icon=BeautifyIcon(
                number=int(row.Population),
                icon_shape='circle',
                background_color='darkred',
                border_color='black',
                text_color='white',
                inner_icon_style='font-size:10px;',
                icon='user'
            )
        ).add_to(pop_layer)

# 3. Add both groups to the map
snap_layer.add_to(m)
pop_layer.add_to(m)

# Step 2: Initialize new layers
agg_marker_layer = FeatureGroup(name='Aggregated Pop (Beautified)')
agg_snap_layer = FeatureGroup(name='Snap lines (aggregated)')

# Step 3: Loop over each aggregated row
for _, row in resnapped.iterrows():
    pop = int(row['Population'])
    centroid = transform(to_wgs84, row['geometry'])
    snapped = transform(to_wgs84, row['nearest_node_agg_geom'])

    # Snap line (dashed)
    line = LineString([centroid, snapped])
    agg_snap_layer.add_child(folium.PolyLine(
        locations=[(lat, lon) for lon, lat in line.coords],
        color='black',
        weight=2,
        dash_array='4,4',
        tooltip=f"Snap: {row['nearest_node_agg_distance']:.1f} m"
    ))

    # Beautify icon
    Marker(
        location=(centroid.y, centroid.x),
        tooltip=f"Total Pop: {pop:,}",
        icon=BeautifyIcon(
            number=pop,
            icon_shape='marker',
            background_color='darkblue',
            border_color='white',
            text_color='white',
            inner_icon_style='font-size:11px;',
            icon='users'
        )
    ).add_to(agg_marker_layer)

# Step 4: Add layers to map
agg_snap_layer.add_to(m)
agg_marker_layer.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
fu.folium_to_png(m, str(images_path / 'detail_roundabout_hdx_pop_snap_agg'))
m

In [ ]:
edgesSubset.highway.value_counts(dropna=False)

In [ ]:
edgesVisible.highway.value_counts(dropna=False)

In [ ]:
transit = iu.load_or_acquire(
    _data_path,
    'nl_transit_ferry',
    osm.get_data_by_custom_criteria,
    logger=logger,
    verbose_args=True,
    custom_filter={
        'route': ['ferry'],
        'public_transport': True
    },
    filter_type="keep"
)     

In [ ]:
transit.explore()